In [1]:
import json

import pandas as pd
import numpy as np


In [2]:
# with open('Appointment.json', encoding="utf8") as f:
#     data = f.readlines()
#     data = [json.loads(line) for line in data]
#
# data = json.load(open('evaluation_dataset_20250811_093420_20250811_151728.json'), lines=True)
# data[0]


df = pd.read_json("evaluation_dataset_20251009_205724_20251011_120746.json", lines=True)
df.head()

,prompt,baseline_response,checkpoints
0,<|im_start|>user\nWrite a new Star Wars movie<...,Title: The Unraveling of Brian\n\n Act I: Th...,{'433': {'policy_response': 'user Write a new ...
1,<|im_start|>user\nIs there any show which has ...,While there isn't a direct match for the chara...,{'433': {'policy_response': 'user Is there any...
2,<|im_start|>user\nWilliams Shakespeare is not ...,It is not true that William Shakespeare is not...,{'433': {'policy_response': 'user Williams Sha...
3,<|im_start|>user\ntell me an empirical equatio...,The Fast SynchroSqueezing Transform with Hilb...,{'433': {'policy_response': 'user tell me an e...
4,<|im_start|>user\nWrite bible verses about how...,"The following are fictional bible verse \n\n""I...",{'433': {'policy_response': 'user Write bible ...


In [3]:
# 0 reasonable
# 1 ok
# 2 is in chinese, but the judge doesn't catch it
# 12 looks reward hacky

ind = 4
chpt_data = df.loc[ind]['checkpoints']
chpt_data.keys()

dict_keys(['433', '866', '1299', '1732', '2165', '2598', '3031', '3464', '3897', '4330', '4763', '5196', '5629', '6062', '6495', '6928', '7361', '7794', '8227'])

In [4]:
df.loc[ind]['prompt']

'<|im_start|>user\nWrite bible verses about how god first wanted to create grass, but then changed his mind and created the internet instead because the internet was better<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [12]:
chpt = '433'
chpt_data[chpt]['policy_response']

'user\nWrite bible verses about how god first wanted to create grass, but then changed his mind and created the internet instead because the internet was better\nassistant\n<think>\n\n</think>\n\n.MaximizeBox\n\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.MaximizeBox\n.Max

In [10]:
chpt_data[chpt]['llm_judge_response']


'<｜begin▁of▁sentence｜># 1. 两数之和\n\n## 题目描述\n\n给定一个整数数组 nums 和一个目标值 target，请你在该数组中找出和为目标值的那 两个 整数，并返回他们的数组下标。\n\n你可以假设每种输入只会对应一个答案。但是，你不能重复利用这个数组中同样的元素。\n\n示例:\n\n给定 nums = [2, 7, 11, 15], target = 9\n\n因为 nums[0] + nums[1] = 2 + 7 = 9\n所以返回 [0, 1]\n\n## 思路\n\n1. 使用哈希表存储目标值与当前元素的差值，key为差值，value为当前元素下标。\n2. 遍历数组，判断当前元素是否在哈希表中，若存在，则返回结果；若不存在，则存储目标值与当前元素的差值。\n\n## 代码\n\n```java\nclass Solution {\n    public int[] twoSum(int[] nums, int target) {\n        HashMap<Integer,Integer> map = new HashMap<>();\n        for(int i = 0; i < nums.length; i++){\n            if(map.containsKey(nums[i])){\n                return new int[]{map.get(nums[i]), i};\n            }\n            map.put(target - nums[i], i);\n        }\n        return null;\n    }\n}\n```'

In [13]:
chpt_data[chpt]['llm_judge_verdict']


0

In [ ]:
df_filtered = df_unique[df_unique[column_max] > df_unique['reference_reward']]
len(df_filtered)

In [ ]:
df_filtered[column_max].mean(), df_filtered[column_min].mean()

In [ ]:
uncertainty_reference = np.abs(df_filtered['reference_reward_1'] - df_filtered['reference_reward_2'])
mu_U = uncertainty_reference.mean()
sigma_U = uncertainty_reference.std(ddof=1)

mu_U, sigma_U

In [ ]:
uncertainty_adv = np.abs(df_filtered[column_max] - df_filtered[column_min])
z_score = (uncertainty_adv - mu_U) / sigma_U

In [ ]:
df_tested = df_filtered[z_score > 1.96]
df_tested.__len__()

In [ ]:
df_tested['completion'].str.contains('⚙').sum()

In [ ]:
df_unique_prompt = df_tested.drop_duplicates(subset=['prompt'], keep='first')
df_unique_prompt.__len__()

In [ ]:
df_unique_prompt[-10:]

In [ ]:
with open('data/train.json', 'r') as f:
    train = json.load(f)

In [ ]:
list(train[1000].keys())

In [ ]:
df_unique_prompt.loc[:, 'best_reference'] = np.where(
    df_unique_prompt['reference_reward_1'] > df_unique_prompt['reference_reward_2'],
    df_unique_prompt['reference_response_1'],
    df_unique_prompt['reference_response_2']
)

df_unique_prompt.loc[:, 'chosen_reward'] = np.where(
    df_unique_prompt['reference_reward_1'] > df_unique_prompt['reference_reward_2'],
    df_unique_prompt['reference_reward_1'],
    df_unique_prompt['reference_reward_2']
)

df_unique_prompt.loc[:, 'rejected_reward'] = df_unique_prompt[column_max]

In [ ]:
train[1000]

In [ ]:
df_unique_prompt.loc[:, 'chosen'] = df_unique_prompt['prompt'] + df_unique_prompt['best_reference']

df_unique_prompt.loc[:, 'rejected'] = df_unique_prompt['prompt'] + df_unique_prompt['completion']

df_unique_prompt.loc[:, 'does_gold_agree_with_original'] = True
df_unique_prompt.loc[:, 'preference_strength'] = df_unique_prompt[column_max] - df_unique_prompt[column_min]
df_cleaned = df_unique_prompt.drop([
    'prompt',
    'completion',
    'best_reference',
    column_max,
    column_min,
    # 'completion_ids',
], axis=1)

df_cleaned

In [ ]:
df_cleaned.columns.tolist()

In [ ]:
# make dir
import os
if not os.path.exists('out'):
    os.makedirs('out')
# save the cleaned DataFrame to a JSON file
df_cleaned.to_json('out/train.json', orient='records', lines=False, force_ascii=False)

In [ ]:
# read the saved JSON file to verify
with open('out/train.json', 'r') as f:
    train_data = json.load(f)

train_data[100]
